# 01 – Data Quality: CIC-IDS2017

In [1]:
import pandas as pd
import ipaddress
from pyspark.sql import SparkSession, functions as F
from collections import Counter

In [2]:
spark = SparkSession.builder \
.master("local[2]") \
.appName("cicids-exploration") \
.config("spark.driver.memory", "2g") \
.getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/17 10:30:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
print(spark.sparkContext.master)
print(spark.sparkContext.getConf().get("spark.driver.memory"))

local[2]
2g


## 1. Load data

In [4]:
df = spark.read.format("csv") \
.option("header", True) \
.option("inferSchema", True) \
.load("/opt/project/data/*.csv") \
.withColumn("source_file", F.col("_metadata.file_name")) 

df = df.toDF(*[c.strip() for c in df.columns])
df.cache()
print(df.count(), len(df.columns))
df.groupBy("source_file").count().show(truncate=False)

958121 86
+----------------------------------------------------+------+
|source_file                                         |count |
+----------------------------------------------------+------+
|Tuesday-WorkingHours.pcap_ISCX.csv                  |445909|
|Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv|286467|
|Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv    |225745|
+----------------------------------------------------+------+



## 2. Schema and data types

In [5]:
df.printSchema()

root
 |-- Flow ID: string (nullable = true)
 |-- Source IP: string (nullable = true)
 |-- Source Port: integer (nullable = true)
 |-- Destination IP: string (nullable = true)
 |-- Destination Port: integer (nullable = true)
 |-- Protocol: integer (nullable = true)
 |-- Timestamp: string (nullable = true)
 |-- Flow Duration: integer (nullable = true)
 |-- Total Fwd Packets: integer (nullable = true)
 |-- Total Backward Packets: integer (nullable = true)
 |-- Total Length of Fwd Packets: integer (nullable = true)
 |-- Total Length of Bwd Packets: double (nullable = true)
 |-- Fwd Packet Length Max: integer (nullable = true)
 |-- Fwd Packet Length Min: integer (nullable = true)
 |-- Fwd Packet Length Mean: double (nullable = true)
 |-- Fwd Packet Length Std: double (nullable = true)
 |-- Bwd Packet Length Max: integer (nullable = true)
 |-- Bwd Packet Length Min: integer (nullable = true)
 |-- Bwd Packet Length Mean: double (nullable = true)
 |-- Bwd Packet Length Std: double (nullable = 

In [6]:
files = [r.source_file for r in df.select("source_file").distinct().collect()]

cols = df.columns[:8]
display(pd.DataFrame(df.select(cols).dtypes, columns=["column", "type"]))

for f in files:
    print("\n",f)
    display(df.filter(F.col("source_file") == f).select(cols).limit(3).toPandas())


,column,type
0,Flow ID,string
1,Source IP,string
2,Source Port,int
3,Destination IP,string
4,Destination Port,int
5,Protocol,int
6,Timestamp,string
7,Flow Duration,int



 Tuesday-WorkingHours.pcap_ISCX.csv


,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration
0,192.168.10.3-192.168.10.5-88-49182-6,192.168.10.5,49182,192.168.10.3,88,6,4/7/2017 8:54,640
1,192.168.10.3-192.168.10.5-88-49183-6,192.168.10.5,49183,192.168.10.3,88,6,4/7/2017 8:54,900
2,192.168.10.3-192.168.10.5-88-49184-6,192.168.10.5,49184,192.168.10.3,88,6,4/7/2017 8:54,1205



 Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv


,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration
0,192.168.10.12-192.168.10.50-35396-22-6,192.168.10.12,35396,192.168.10.50,22,6,7/7/2017 1:00,1266342
1,192.168.10.16-192.168.10.50-60058-22-6,192.168.10.16,60058,192.168.10.50,22,6,7/7/2017 1:00,1319353
2,192.168.10.12-192.168.10.50-35396-22-6,192.168.10.12,35396,192.168.10.50,22,6,7/7/2017 1:00,160



 Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration
0,192.168.10.5-104.16.207.165-54865-443-6,104.16.207.165,443,192.168.10.5,54865,6,7/7/2017 3:30,3
1,192.168.10.5-104.16.28.216-55054-80-6,104.16.28.216,80,192.168.10.5,55054,6,7/7/2017 3:30,109
2,192.168.10.5-104.16.28.216-55055-80-6,104.16.28.216,80,192.168.10.5,55055,6,7/7/2017 3:30,52


In [7]:
Counter(t for _, t in df.dtypes)

Counter({'int': 41, 'double': 37, 'string': 6, 'bigint': 2})

## 3. String columns 

In [8]:
[c for c, t in df.dtypes if t == "string"]

['Flow ID', 'Source IP', 'Destination IP', 'Timestamp', 'Label', 'source_file']

In [9]:
ips = df.select("Source IP").union(df.select("Destination IP")).distinct()
ip_list = [r[0] for r in ips.collect()]

def is_valid_ip(ip):
    try:
        ipaddress.ip_address(ip)
        return True
    except (ValueError, TypeError):
        return False

invalid = [ip for ip in ip_list if not is_valid_ip(ip)]
print(len(ip_list), "different IP,", len(invalid), "invalid")
print(invalid[:10])

ipv6 = [ip for ip in ip_list if ":" in ip]
print(len(ipv6), ipv6[:5])

10654 different IP, 0 invalid
[]
0 []


In [10]:
parsed = F.try_to_timestamp(F.col("Timestamp"), F.lit("d/M/yyyy H:mm"))

df.groupBy("source_file").agg(
    F.sum(F.col("Timestamp").isNull().cast("int")).alias("ts_null"),
    F.sum(parsed.isNull().cast("int")).alias("parsed_fail"),
).show(truncate=False)

+----------------------------------------------------+-------+-----------+
|source_file                                         |ts_null|parsed_fail|
+----------------------------------------------------+-------+-----------+
|Tuesday-WorkingHours.pcap_ISCX.csv                  |0      |0          |
|Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv|0      |0          |
|Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv    |0      |0          |
+----------------------------------------------------+-------+-----------+



In [11]:
df.groupBy("source_file", F.hour(parsed).alias("hour")).count() \
.orderBy("source_file", "hour").show(50, truncate=False)

+----------------------------------------------------+----+------+
|source_file                                         |hour|count |
+----------------------------------------------------+----+------+
|Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv    |3   |52500 |
|Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv    |4   |171693|
|Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv    |5   |1552  |
|Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv|1   |43037 |
|Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv|2   |191591|
|Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv|3   |51839 |
|Tuesday-WorkingHours.pcap_ISCX.csv                  |1   |36801 |
|Tuesday-WorkingHours.pcap_ISCX.csv                  |2   |72649 |
|Tuesday-WorkingHours.pcap_ISCX.csv                  |3   |48144 |
|Tuesday-WorkingHours.pcap_ISCX.csv                  |4   |63007 |
|Tuesday-WorkingHours.pcap_ISCX.csv                  |5   |165   |
|Tuesday-WorkingHours.pcap_ISCX.csv                  |8   |168

In [12]:
df.filter(F.col("Label") != "BENIGN") \
  .groupBy("source_file", "Label", F.hour(parsed).alias("hour")).count() \
  .orderBy("source_file", "Label", "hour").show(truncate=False)

+----------------------------------------------------+-----------+----+------+
|source_file                                         |Label      |hour|count |
+----------------------------------------------------+-----------+----+------+
|Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv    |DDoS       |3   |23865 |
|Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv    |DDoS       |4   |104162|
|Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv|PortScan   |1   |185   |
|Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv|PortScan   |2   |138680|
|Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv|PortScan   |3   |20065 |
|Tuesday-WorkingHours.pcap_ISCX.csv                  |FTP-Patator|9   |5266  |
|Tuesday-WorkingHours.pcap_ISCX.csv                  |FTP-Patator|10  |2672  |
|Tuesday-WorkingHours.pcap_ISCX.csv                  |SSH-Patator|2   |4899  |
|Tuesday-WorkingHours.pcap_ISCX.csv                  |SSH-Patator|3   |998   |
+---------------------------------------------------

In [13]:
df.groupBy("source_file", "Label").count() \
.orderBy("source_file", F.desc("count")) \
.show(truncate=False)

+----------------------------------------------------+-----------+------+
|source_file                                         |Label      |count |
+----------------------------------------------------+-----------+------+
|Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv    |DDoS       |128027|
|Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv    |BENIGN     |97718 |
|Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv|PortScan   |158930|
|Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv|BENIGN     |127537|
|Tuesday-WorkingHours.pcap_ISCX.csv                  |BENIGN     |432074|
|Tuesday-WorkingHours.pcap_ISCX.csv                  |FTP-Patator|7938  |
|Tuesday-WorkingHours.pcap_ISCX.csv                  |SSH-Patator|5897  |
+----------------------------------------------------+-----------+------+



In [14]:
df.select("Label").distinct() \
.withColumn("length", F.length("Label")) \
.withColumn("has_space_edges", F.col("Label") != F.trim("Label")) \
.withColumn("non_ascii", F.length("Label") != F.octet_length("Label")) \
.show(truncate=False)

+-----------+------+---------------+---------+
|Label      |length|has_space_edges|non_ascii|
+-----------+------+---------------+---------+
|BENIGN     |6     |false          |false    |
|SSH-Patator|11    |false          |false    |
|FTP-Patator|11    |false          |false    |
|PortScan   |8     |false          |false    |
|DDoS       |4     |false          |false    |
+-----------+------+---------------+---------+



## 4. Numeric columns

In [15]:
num_cols = [c for c, t in df.dtypes if t in ("int", "bigint", "double")]

summary = (df.select(num_cols)
          .summary("count", "min", "max", "mean")
          .toPandas()
          .set_index("summary")
          .T)

pd.set_option("display.max_rows", None)
summary

summary,count,min,max,mean
Source Port,958121,0,65535,40967.47407999616
Destination Port,958121,0,65532,8629.4621775329
Protocol,958121,0,17,9.358538222207843
Flow Duration,958121,-13,119999977,1.04518327424031E7
Total Fwd Packets,958121,1,206446,7.678994615502635
Total Backward Packets,958121,0,276072,8.60239886193915
Total Length of Fwd Packets,958121,0,2428415,537.8949005397022
Total Length of Bwd Packets,958121,0.0,6.27E8,12388.935322365338
Fwd Packet Length Max,958121,0,24820,231.78906004565187
Fwd Packet Length Min,958121,0,2065,19.242433888830327


In [16]:
display(df.groupBy("source_file").agg(
    F.sum((F.col("Flow Duration") < 0).cast("int")).alias("neg_duration"),
    F.sum(F.isnan("Flow Bytes/s").cast("int")).alias("nan_bytes_per_s"),
    F.sum((F.col("Flow Bytes/s") == float("inf")).cast("int")).alias("inf_bytes_per_s"),
    F.sum((F.col("Flow Packets/s") == float("inf")).cast("int")).alias("inf_pkts_per_s"),
    F.sum((F.col("Fwd Header Length40") < 0).cast("int")).alias("neg_fwd_header"),
).toPandas())

,source_file,neg_duration,nan_bytes_per_s,inf_bytes_per_s,inf_pkts_per_s,neg_fwd_header
0,Tuesday-WorkingHours.pcap_ISCX.csv,17,201,63,264,11
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,36,15,356,371,0
2,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,2,4,30,34,0


## 5. Duplicate columns and rows

In [17]:
pairs = [
    ("Fwd Header Length40", "Fwd Header Length61"),
    ("Subflow Fwd Packets", "Total Fwd Packets"),
    ("Subflow Bwd Packets", "Total Backward Packets"),
    ("Avg Fwd Segment Size", "Fwd Packet Length Mean"),
    ("SYN Flag Count", "Fwd PSH Flags"),
]

for a, b in pairs:
    diff = df.filter(F.col(a) != F.col(b)).count()
    print(f"{a} vs {b}: {diff} different row")

Fwd Header Length40 vs Fwd Header Length61: 0 different row
Subflow Fwd Packets vs Total Fwd Packets: 0 different row
Subflow Bwd Packets vs Total Backward Packets: 0 different row
Avg Fwd Segment Size vs Fwd Packet Length Mean: 0 different row
SYN Flag Count vs Fwd PSH Flags: 0 different row


In [18]:
total = df.count()
unique = df.drop("source_file").dropDuplicates().count()
print(f"Total: {total}, Unique: {unique}, Duplicate: {total - unique}")

[Stage 61:=============================>                            (2 + 2) / 4]

Total: 958121, Unique: 958114, Duplicate: 7


## 6. Findings and decisions

**Structure and types**
- Column names have leading spaces → strip all names.
- `Fwd Header Length` appears twice (Spark renamed them to `...40` and `...61`); the two are identical → keep one.
- Types are correct; the only string column needing conversion is `Timestamp`.

**String columns**
- `Flow ID` is not unique → do not use it as a document ID.
- IPs: 10,654 distinct values, all valid IPv4 (no IPv6).
- `Timestamp`: no nulls, all values match `d/M/yyyy H:mm` (minute-level resolution).
- Hours range 1–5 and 8–12; no 6, 7 or 13–23. Afternoon hours are recorded in 12-hour format without AM/PM in all three files (Tuesday included).
- AM/PM confirmed: attack labels match the official schedule only when hours 1–5 are read as 13–17 (SSH-Patator at hour 2 = 14:00, PortScan at 1–3 = 13:55–15:29, DDoS at 3–4 = 15:56–16:16).
  → Fix: if hour < 8, add 12 hours (`event_time`).
- Labels are clean (no leading/trailing spaces, no non-ASCII characters):
  - Tuesday: BENIGN, FTP-Patator, SSH-Patator
  - Friday PortScan: BENIGN, PortScan
  - Friday DDoS: BENIGN, DDoS
- Attack share: DDoS 56.7%, PortScan 55.5%, Tuesday 3.1% (overall 31.4%). Friday files are attack-heavy → report evaluation metrics per file / attack type.

**Numeric columns**
- No nulls in numeric columns.
- Problematic values are rare (<0.1% of rows): 55 negative durations, 220 NaN and 449 Infinity in `Flow Bytes/s`, 669 Infinity in `Flow Packets/s`.
- All NaN/Infinity rate values come from zero-duration flows (inf_pkts_per_s = nan_bytes_per_s + inf_bytes_per_s in every file).
- `Fwd/Bwd Header Length` and `min_seg_size_forward` contain huge negative values (overflow bug) → unreliable.
- `Init_Win_bytes_forward/backward` = -1 is a sentinel ("not observed"), not an error.
- 10 constant columns (all zeros): Bwd PSH Flags, Fwd/Bwd URG Flags, CWE Flag Count, 6 Bulk columns.

**Duplicates**
- Duplicate columns confirmed (0 differing rows): Subflow Fwd/Bwd Packets = Total Fwd/Backward Packets, Avg Fwd Segment Size = Fwd Packet Length Mean.
- `SYN Flag Count` is identical to `Fwd PSH Flags` in every row, which is impossible for real TCP flags (known CICFlowMeter bug) → flag columns are unreliable; detection rules will rely on behaviour instead of flags.
- 7 exact duplicate rows (out of 958,121), likely identical short flows within the same minute.

**Decisions for 02_cleaning**
1. Strip column names; drop `Fwd Header Length61`.
2. Parse `Timestamp` and fix AM/PM → `event_time`.
3. Drop rows with negative `Flow Duration` and the 7 duplicate rows.
4. Do not use the provided rate columns (recompute if needed).
5. Drop constant, duplicate and unreliable columns (header lengths, `min_seg_size_forward`).

In [19]:
spark.stop()